# Your First Quantum Programs 🚀
> **⚠️ THIS NOTEBOOK RUNS ON GOOGLE COLAB.** If you are reading this anywhere else (VS Code, Jupyter, a downloaded file), go to [colab.research.google.com](https://colab.research.google.com), choose **File → Upload notebook**, and open this file there. Colab is free and needs only a Google account.
### Hands-on workshop · Satyug Darshan Institute of Engineering and Technology · 12 August 2026
**Manisha Malhotra** · Visiting Researcher, Imperial College London

Welcome! In the next 40 minutes your pod will:
1. Put a qubit in superposition and measure it
2. Entangle two qubits
3. Break a quantum computer on purpose (with maths, not hammers)
4. Find the energy of a real molecule with the same algorithm I use in my research
5. Submit your pod's first job to a REAL quantum computer in Finland

**Pod rules:** groups of 3, one laptop, rotate the driver every cell. `Shift+Enter` runs a cell. Look for the `# >>> CHANGE THIS <<<` line in each exercise.


---
## Cell 0 · Setup (run this FIRST, right now)
This installs Qrisp, the quantum programming toolkit we will use. It takes about 2 minutes. Run it and listen to the talk while it works.

In [ ]:
# Run me first! (about 2 minutes)
import sys
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    print("⚠️  You are NOT in Google Colab. This notebook is designed for Colab.")
    print("    Go to https://colab.research.google.com → File → Upload notebook → pick this file.")
    print("    (It will still try to install below, but Colab is the supported path.)\n")

%pip install qrisp --quiet
print("\n✅ Setup complete. Your pod is ready.")

---
## Cell 1 · Your first qubit 🪙
A classical bit is 0 **or** 1. A qubit can be 0 **and** 1 at the same time, until you measure it.

The gate `h` (Hadamard) puts a qubit into an equal blend of 0 and 1. Then we measure it 1,000 times. If quantum mechanics is right, you should see roughly 50% zeros and 50% ones, even though we run the *identical* circuit every time.

**Driver 1, you're up.** Run the cell, then change the number of shots and run again. Does the split get closer to 50/50 with more shots?

In [ ]:
from qrisp import QuantumVariable, h

qv = QuantumVariable(1)   # one fresh qubit, starts at 0
h(qv[0])                  # put it in superposition: 0 AND 1

shots = 1000              # >>> CHANGE THIS <<<  try 10, then 100, then 100000
counts = qv.get_measurement(shots=shots)

print(f"Results from {shots} measurements of the SAME circuit:")
for outcome, fraction in sorted(counts.items()):
    bar = "█" * int(fraction * 40)
    print(f"  |{outcome}⟩  {fraction:6.1%}  {bar}")

**What just happened?** Every single run was identical, yet the answers are random. That randomness is not a bug or missing information: it is how nature works at this scale. With few shots the split is ragged; with many shots it settles to 50/50. Quantum computing is a statistics game.


---
## Cell 2 · Entangle two qubits 🔗
Now the famous one. We link two qubits so that each one *alone* looks random, but they **always agree**. Einstein called this "spooky action at a distance". You are about to build it in four lines.

**Rotate the driver.** Run it, then try the CHANGE THIS experiment.

In [ ]:
from qrisp import QuantumVariable, h, cx

qv = QuantumVariable(2)   # two fresh qubits: |00⟩
h(qv[0])                  # qubit 0 into superposition
# h(qv[1])                # (leave this commented for now)
cx(qv[0], qv[1])          # ENTANGLE: qubit 1 copies qubit 0's fate

# >>> CHANGE THIS <<<  Two experiments, run after each change:
#  1. Put a # in front of the cx line. Which outcomes appear now? Do the qubits still agree?
#  2. Now ALSO remove the # from the h(qv[1]) line. Both qubits are random. Are they correlated?
#  Then restore the original circuit.

counts = qv.get_measurement(shots=1000)
print("Measured outcomes (qubit0, qubit1):")
for outcome, fraction in sorted(counts.items()):
    bar = "█" * int(fraction * 40)
    print(f"  |{outcome}⟩  {fraction:6.1%}  {bar}")

**What just happened?** With the `cx` in place you only ever see `00` and `11`: which one is random, but the qubits **always agree**.

Experiment 1 (no `cx`): you see `00` and `10`. Qubit 0 is still random, but qubit 1 is frozen at 0, ignoring qubit 0 completely. The agreement is gone.

Experiment 2 (`h` on both, no `cx`): all four outcomes appear equally. Both qubits are random, but knowing one tells you **nothing** about the other.

Randomness is easy. *Correlated* randomness, agreement without any communication, is entanglement, and it is the resource that makes quantum computers powerful.


---
## Cell 3 · Break it on purpose 💥
Real quantum hardware is noisy: every gate is only ~99% accurate. Sounds fine, right?

Here is the catch: errors **multiply**. A circuit with $n$ gates survives with probability roughly $0.99^n$.

This is exactly how my first real-hardware experiment died. My 14-qubit circuit needed **418** two-qubit gates once it was mapped onto the chip. Run the cell to see what the machine saw.

**Rotate the driver.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gate_accuracy = 0.99

# >>> CHANGE THIS <<<  my failed circuit had 418 gates. My fixed one had 24. Try both!
n_gates = 418

survival = gate_accuracy ** n_gates   # chance the quantum state survives the whole circuit

# What a perfect Bell state measurement looks like vs what noise leaves behind
ideal = {"00": 0.5, "01": 0.0, "10": 0.0, "11": 0.5}
noisy = {k: survival * v + (1 - survival) * 0.25 for k, v in ideal.items()}  # noise scrambles toward all-equal

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
ax[0].bar(ideal.keys(), ideal.values(), color="#1B2447"); ax[0].set_title("Perfect machine")
ax[1].bar(noisy.keys(), noisy.values(), color="#E8A33D"); ax[1].set_title(f"Real machine, {n_gates} gates")
ax[0].set_ylabel("probability")
plt.suptitle(f"Survival probability: {gate_accuracy}^{n_gates} = {survival:.1%}")
plt.tight_layout(); plt.show()

print(f"With {n_gates} gates, only {survival:.1%} of your quantum information survives.")
print("The rest is scrambled into equal random noise, a 'maximally mixed state'.")

**What just happened?** At 418 gates only ~1.5% of the signal survives: the bars flatten into useless uniform noise. That flat picture is exactly what a quantum computer in Finland sent back to me last year. At 24 gates (my redesigned circuit) about 79% survives and the answer is clearly visible. **Lesson: on today's hardware, the shortest circuit wins.** Frontier research in this field is largely the art of doing more with fewer gates.


---
## Cell 4 · Find a molecule's energy ⚛️ (the real thing)
This is a genuine **VQE** (Variational Quantum Eigensolver): the same algorithm I run on iridium oxide, here on the smallest real molecule, **H₂** (hydrogen).

The idea:
1. Qubits represent where the electrons live
2. A knob `theta` adjusts the electron arrangement
3. A classical optimiser turns the knob until the energy is as low as possible
4. The lowest energy is the molecule's ground state: real chemistry

**Rotate the driver**, run it, and watch the optimiser learn.

In [ ]:
from qrisp import QuantumVariable, x, ry, cx
from qrisp.operators.qubit import X, Z
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# The hydrogen molecule, written in the language of qubits (a "Hamiltonian").
# These numbers come from real quantum chemistry for H2 at its natural bond length.
H = (-1.052373245772859
     + 0.39793742484318045 * Z(0)
     - 0.39793742484318045 * Z(1)
     - 0.01128010425623538 * Z(0) * Z(1)
     + 0.18093119978423156 * X(0) * X(1))

def prepare(theta):
    qv = QuantumVariable(2)
    x(qv[0])            # start from the textbook guess (both electrons in the lowest orbital)
    ry(theta, qv[1])    # the knob: move some electron amplitude to the higher orbital
    cx(qv[1], qv[0])    # keep the electron bookkeeping consistent
    return qv

energy = H.expectation_value(prepare, precision=0.001)

history = []
def cost(params):
    e = float(energy(float(params[0])))
    history.append(e)
    print(f"  step {len(history):2d}: theta = {float(params[0]):+.3f}  →  energy = {e:.5f} hartree")
    return e

theta_start = 0.5   # >>> CHANGE THIS <<<  try 3.0 or -2.0. Does it still find the answer?
print("The optimiser starts turning the knob...")
result = minimize(cost, [theta_start], method="COBYLA", options={"maxiter": 40})

exact = H.ground_state_energy()
plt.figure(figsize=(8, 4))
plt.plot(history, "o-", color="#1B2447", label="VQE energy")
plt.axhline(exact, color="#E8A33D", ls="--", label=f"true answer: {exact:.5f}")
plt.xlabel("optimiser step"); plt.ylabel("energy (hartree)")
plt.title("A quantum computer learning the energy of a hydrogen molecule")
plt.legend(); plt.tight_layout(); plt.show()

print(f"\nVQE found : {result.fun:.5f} hartree")
print(f"True value: {exact:.5f} hartree")
print("You just did computational quantum chemistry. 🎉")

**What just happened?** The optimiser twiddled one knob and homed in on the true ground-state energy of hydrogen. My research is this exact loop, scaled up: 7 qubits instead of 2, a catalyst surface instead of H₂, and a real superconducting chip instead of a simulator, where Cell 3's noise problem is the daily battle.


---
## Stretch cell · For fast pods 🏎️
A GHZ state entangles **many** qubits at once: all zeros or all ones, nothing in between. How many qubits can your laptop simulate before it slows down? (Every extra qubit doubles the memory needed. This is exactly why we want quantum hardware.)

In [ ]:
from qrisp import QuantumVariable, h, cx

n = 5   # >>> CHANGE THIS <<<  try 10, 15, 20, 24... when does your laptop start sweating?

qv = QuantumVariable(n)
h(qv[0])
for i in range(n - 1):
    cx(qv[i], qv[i + 1])

counts = qv.get_measurement(shots=1000)
print(f"GHZ state on {n} qubits:")
for outcome, fraction in sorted(counts.items()):
    print(f"  |{outcome}⟩  {fraction:6.1%}")
print(f"\n{n} entangled qubits track 2^{n} = {2**n:,} possibilities at once.")

---
## Cell 5 · Fire at Finland 🇫🇮 (live, together)
Everything above ran on a simulator. Now your pod submits a **real job to a real superconducting quantum computer**.

**In the room, on the timer:**
1. Everyone: create a free account at [resonance.iqm.tech](https://resonance.iqm.tech) using Google sign-in (works on your phone)
2. Driver only: on your dashboard, generate an **API token** and paste it below on your pod's laptop
3. Check the dashboard for an available device name (for example `garnet` or `emerald`) and fill it in
4. Set `RUN_ON_HARDWARE = True` and run the cell: your job goes into the queue in Finland

**The queue is shared with researchers worldwide, so results may take minutes or hours.** If the cell is still waiting after a couple of minutes, press the stop button: your job is safely queued, and your results will be on your Resonance dashboard when they're done. Comparing them to the simulator is tonight's homework, and everyone who wasn't the driver runs their own job at home with their own token.

**🔑 Token hygiene, week-one lesson:** a token is a password. Driver, delete it from the cell after running. Never screenshot it, commit it to GitHub, or share it.

In [ ]:
# CELL 5: real hardware. Needs an IQM Resonance account + API token (see steps above).
%pip install "qrisp[iqm]" --quiet

RUN_ON_HARDWARE = False   # set to True once the token and device below are filled in

if RUN_ON_HARDWARE:
    from qrisp.interface import IQMBackend
    from qrisp import QuantumVariable, h, cx

    backend = IQMBackend(
        token="PASTE_YOUR_API_TOKEN_HERE",   # driver's token. DELETE after running. Never share.
        device_instance="garnet",            # check your Resonance dashboard for available devices
    )

    qv = QuantumVariable(2)
    h(qv[0])
    cx(qv[0], qv[1])

    print("Submitting your Bell state to a quantum computer near Helsinki...")
    print("(If this waits more than a couple of minutes: press stop. Your job is queued;")
    print(" results will appear on your Resonance dashboard when it runs.)\n")

    counts = qv.get_measurement(backend=backend, shots=1000)

    print("Results from REAL quantum hardware:")
    for outcome, fraction in sorted(counts.items()):
        bar = "█" * int(fraction * 40)
        print(f"  |{outcome}⟩  {fraction:6.1%}  {bar}")
    print("\nCompare with Cell 2: it is NOT a perfect 50/50 between 00 and 11.")
    print("Those stray 01 and 10 counts are real hardware noise: Cell 3, live.")
else:
    print("Set RUN_ON_HARDWARE = True after adding the driver's token and device name.")

---
## Where to go next 🗺️
- **Learn:** [IQM Academy](https://www.iqmacademy.com) · [Qrisp documentation](https://qrisp.eu)
- **Today's code and my research pipeline:** [github.com/Codexee/Iridium-Oxide-ASE](https://github.com/Codexee/Iridium-Oxide-ASE)
- **My preprint** (the research behind this workshop): [researchsquare.com/article/rs-9495573/v1](https://www.researchsquare.com/article/rs-9495573/v1)

*Fourteen years ago I sat where you are sitting. What you do next is the interesting part.*

— Manisha Malhotra, August 2026